In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
import glob
image_paths = glob.glob("/kaggle/input/q3-stage3-2026/dataset/images/*.*")
image_paths = sorted(image_paths)
mask_paths = glob.glob("/kaggle/input/q3-stage3-2026/dataset/masks/*.*")
mask_paths = sorted(mask_paths)
print(image_paths[:5])
print(mask_paths[:5])

In [ ]:
# TO DO
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import glob
# images_paths = glob.glob("/kaggle/input/q3-stage3-2026/dataset/images/*.jpg")
# masks_paths = glob.glob("/kaggle/input/q3-stage3-2026/dataset/masks/*.png")
# print(images_paths[:5])
# print(masks_paths[:5])
# Custom Dataset Class for Underwater Segmentation
class UnderwaterSegmentationDataset(Dataset):

    def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):

        self.image_paths = image_paths
        self.mask_paths = mask_paths


        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Remap mask values to consecutive range [0, num_classes-1]
        mask = remap_mask(mask)

        return image, mask

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep mask values intact
    transforms.PILToTensor(),
])

train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.2, random_state=42
)



train_dataset = UnderwaterSegmentationDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = UnderwaterSegmentationDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)
# Create DataLoaders
batch_size = 8 # time is running out :(
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

# Function to denormalize images for visualization
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose(1, 2, 0)  # Convert from CHW to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip to valid range
    return img

# Display sample images with masks
class_names = ['Background', 'Human divers', 'Plants/sea-grass', 'Wrecks/ruins',
               'Robots', 'Reefs/invertebrates', 'Fish/vertebrates', 'Sea-floor/rocks']

for i in range(3):
    img, mask = train_dataset[i]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Display image
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")

    # Display mask with color map
    axes[1].imshow(mask.squeeze(), cmap='tab10', vmin=0, vmax=7)
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# TO DO
# Install segmentation_models_pytorch
!pip install -q segmentation_models_pytorch

import segmentation_models_pytorch as smp

# Define U-Net Model with pretrained EfficientNet-B1 encoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # 8 classes for multi-class segmentation
).to(device)

print(f"Using device: {device}")
print(f"Model created with encoder: efficientnet-b1")
print(f"Number of classes: 8")

In [ ]:
# TO DO
import torch.optim as optim
from tqdm import tqdm

# Training loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader, desc="Training"):
        images, masks = images.to(device), masks.to(device).long().squeeze(1)  # Squeeze mask to [B, H, W]

        outputs = model(images)  # [B, num_classes, H, W]
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# Validation loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).long().squeeze(1)  # Squeeze mask to [B, H, W]

            outputs = model(images)  # [B, num_classes, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch.nn as nn

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class segmentation
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10
train_losses = []
val_losses = []

# Training loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

print("\nTraining Complete!")

# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# TO DO
import random

# Function to visualize predictions
def visualize_predictions(model, dataset, device, num_samples=5):
    model.eval()

    # Get random test samples
    test_samples = random.sample(range(len(dataset)), num_samples)

    for idx in test_samples:
        img, mask = dataset[idx]

        with torch.no_grad():
            # Forward pass
            pred_mask = model(img.unsqueeze(0).to(device))  # [1, num_classes, H, W]
            pred_mask = torch.argmax(pred_mask, dim=1).cpu().squeeze().numpy()  # Get class with highest probability

        # Prepare ground truth mask
        gt_mask = mask.squeeze().numpy()

        # Display results
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original Image (Denormalized)
        axes[0].imshow(denormalize(img))
        axes[0].set_title("Original Image")
        axes[0].axis("off")

        # Ground Truth Mask
        axes[1].imshow(gt_mask, cmap='tab10', vmin=0, vmax=7)
        axes[1].set_title("Ground Truth Mask")
        axes[1].axis("off")

        # Predicted Mask
        axes[2].imshow(pred_mask, cmap='tab10', vmin=0, vmax=7)
        axes[2].set_title("Predicted Mask")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

# Visualize predictions
visualize_predictions(model, test_dataset, device, num_samples=5)